In [63]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

FIRST_STAGE = "first_stage"
SECOND_STAGE = "second_stage"
FINAL_STAGE = "final_stage"
process_status = {"is_interruption":True}

In [64]:
class MsgState(MessagesState):
    user_request:str
    response:str

In [65]:
def process_first_stage(state: MsgState):
    print("\nFirst Stage Process Begins...")
    state["user_request"] = state["user_request"] + "- Stage One"
    state["response"] = f"{state["user_request"]} - Clearance from First stage process"
    print(state)
    return state

def process_second_stage(state: MsgState):
    print("\nSecond Stage Process Begins...")
    state["user_request"] = state["user_request"] + "- Stage Two"
    if process_status["is_interruption"]:
        process_status["is_interruption"] = False
        print("\nProcess Interrupted")
        raise ValueError("interrupted due to raw material mix")
    state["response"] = f"{state["user_request"]} - Clearance from Second stage process"
    print(state)
    return state

def process_final_stage(state:MsgState):
    print("\nFinal Stage Process Begins...")
    state["user_request"] = state["user_request"] + "- Process Completed"
    state["response"] = f"{state["user_request"]} - Final Clearance -- Product is ready"
    print(state)
    return state

In [66]:
wf_graph = StateGraph(MsgState)
wf_graph.add_node(FIRST_STAGE, process_first_stage)
wf_graph.add_node(SECOND_STAGE, process_second_stage)
wf_graph.add_node(FINAL_STAGE, process_final_stage)

wf_graph.add_edge(START, FIRST_STAGE)
wf_graph.add_edge(FIRST_STAGE, SECOND_STAGE)
wf_graph.add_edge(SECOND_STAGE, FINAL_STAGE)
wf_graph.add_edge(FINAL_STAGE, END)

in_memory_checkpoint = InMemorySaver()
graph = wf_graph.compile(in_memory_checkpoint)

### Checkpointer with Configurable - Thread Id - User-1

In [69]:
configs = {"configurable":{"thread_id":"user-1"}}
#response = graph.invoke({"messages":[{"user_request":"Refine Oil"}]}, configs)
#response = graph.invoke({"messages":[{"role":"user", "content":"Refine Oil"}]}, configs)
response = graph.invoke({"user_request":"Refine Oil"}, config=configs)
print(response)


First Stage Process Begins...
{'messages': [], 'user_request': 'Refine Oil- Stage One', 'response': 'Refine Oil- Stage One - Clearance from First stage process'}

Second Stage Process Begins...
{'messages': [], 'user_request': 'Refine Oil- Stage One- Stage Two', 'response': 'Refine Oil- Stage One- Stage Two - Clearance from Second stage process'}

Final Stage Process Begins...
{'messages': [], 'user_request': 'Refine Oil- Stage One- Stage Two- Process Completed', 'response': 'Refine Oil- Stage One- Stage Two- Process Completed - Final Clearance -- Product is ready'}
{'messages': [], 'user_request': 'Refine Oil- Stage One- Stage Two- Process Completed', 'response': 'Refine Oil- Stage One- Stage Two- Process Completed - Final Clearance -- Product is ready'}


### Checkpointer with Configurable - Thread Id - User-2

In [68]:
configs = {"configurable":{"thread_id":"user-2"}}
response = graph.invoke({"user_request":"Refine Palm Oil"}, config=configs)
print(response)


First Stage Process Begins...
{'messages': [], 'user_request': 'Refine Palm Oil- Stage One', 'response': 'Refine Palm Oil- Stage One - Clearance from First stage process'}

Second Stage Process Begins...
{'messages': [], 'user_request': 'Refine Palm Oil- Stage One- Stage Two', 'response': 'Refine Palm Oil- Stage One- Stage Two - Clearance from Second stage process'}

Final Stage Process Begins...
{'messages': [], 'user_request': 'Refine Palm Oil- Stage One- Stage Two- Process Completed', 'response': 'Refine Palm Oil- Stage One- Stage Two- Process Completed - Final Clearance -- Product is ready'}
{'messages': [], 'user_request': 'Refine Palm Oil- Stage One- Stage Two- Process Completed', 'response': 'Refine Palm Oil- Stage One- Stage Two- Process Completed - Final Clearance -- Product is ready'}
